# 了解数据结构-数据探索

In [1]:
import pandas as pd
data = pd.read_excel('../data/raw/online_retail_II.xlsx',sheet_name=None)
print(type(data))

<class 'dict'>


In [25]:
df = pd.concat(data.values(),ignore_index=True)
print(df.shape)
print(df.info())
print(df.describe())
print(df.head(10))


(1067371, 8)
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 78.8+ MB
None
           Quantity                 InvoiceDate         Price    Customer ID
count  1.067371e+06                     1067371  1.067371e+06  824364.000000
mean   9.938898e+00  2011-01-02 21:13:55.394029  4.649388e+00   15324.638504
min   -8.099500e+04         2009-12-01 07:45:00 -5

### 产看数据缺失值信息

In [17]:
# 统计每个字段的缺失值数量
missing_count = df.isnull().sum()
# print(missing_count)
# 缺失率
missing_rate = df.isnull().mean()*100
# print(missing_rate)

# 整理结果
missing_info = pd.DataFrame({
    '缺失值数量':missing_count,
    '缺失率(%)':missing_rate,
})
display(missing_info)

,缺失值数量,缺失率(%)
Invoice,0,0.000000
StockCode,0,0.000000
Description,4382,0.410541
Quantity,0,0.000000
InvoiceDate,0,0.000000
Price,0,0.000000
Customer ID,243007,22.766873
Country,0,0.000000
Is_Cancelled,0,0.000000


### 统计重复行数量

In [18]:
duplicate_count = df.duplicated().sum()
print(f"数据总行数：{len(df)}")
print(f"重复行数量：{duplicate_count}")
print(f"重复率：{duplicate_count / len(df) * 100:.2f}%")

数据总行数：1067371
重复行数量：34335
重复率：3.22%


### 统计唯一订单数、商品数、客户数和国家数

In [19]:
summary = {
    '数据总行数': len(df),
    '字段数量': len(df.columns) - 1,
    '唯一订单数': df['Invoice'].nunique(),
    '商品数量': df['StockCode'].nunique(),
    '客户数量': df['Customer ID'].nunique(),
    '国家/地区数量': df['Country'].nunique()
}

summary_df = pd.DataFrame(
    list(summary.items()),
    columns=['统计指标', '统计结果']
)

display(summary_df)

,统计指标,统计结果
0,数据总行数,1067371
1,字段数量,8
2,唯一订单数,53628
3,商品数量,5305
4,客户数量,5942
5,国家/地区数量,43


### 检查日期范围，数量和单价的分布，以及负数，零值和极端值

In [8]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'],errors='coerce')
print(f'订单日期范围为：{df['InvoiceDate'].min()}--{df['InvoiceDate'].max()}')
date_null_sum = df['InvoiceDate'].isnull().sum()
print(f'日期缺失数量：{date_null_sum}')

print('Quantity 描述性统计：')

print(df['Quantity'].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
))

print('负数数量：', (df['Quantity'] < 0).sum())
print('零值数量：', (df['Quantity'] == 0).sum())
print('正数数量：', (df['Quantity'] > 0).sum())

订单日期范围为：2009-12-01 07:45:00--2011-12-09 12:50:00
日期缺失数量：0
Quantity 描述性统计：
count    1.067371e+06
mean     9.938898e+00
std      1.727058e+02
min     -8.099500e+04
1%      -3.000000e+00
5%       1.000000e+00
25%      1.000000e+00
50%      3.000000e+00
75%      1.000000e+01
95%      3.000000e+01
99%      1.000000e+02
max      8.099500e+04
Name: Quantity, dtype: float64
负数数量： 22950
零值数量： 0
正数数量： 1044421


### 检查price分布

In [10]:
print('Price 描述性统计：')
print(df['Price'].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
))
print('负单价数量',(df['Price'] < 0).sum())
print('零单价数量：', (df['Price'] == 0).sum())
print('正单价数量：', (df['Price'] > 0).sum())

Price 描述性统计：
count    1.067371e+06
mean     4.649388e+00
std      1.235531e+02
min     -5.359436e+04
1%       2.100000e-01
5%       4.200000e-01
25%      1.250000e+00
50%      2.100000e+00
75%      4.150000e+00
95%      9.950000e+00
99%      1.800000e+01
max      3.897000e+04
Name: Price, dtype: float64
负单价数量 5
零单价数量： 6202
正单价数量： 1061164


### 检查极端值

In [11]:
def check_outliers(series, name):

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = series[
        (series < lower) | (series > upper)
    ]

    print(f'\n{name} 极端值检查')
    print(f'下界：{lower}')
    print(f'上界：{upper}')
    print(f'极端值数量：{len(outliers)}')
    print(f'极端值占比：{len(outliers) / series.notna().sum() * 100:.2f}%')

    return outliers


quantity_outliers = check_outliers(
    df['Quantity'].dropna(),
    'Quantity'
)

price_outliers = check_outliers(
    df['Price'].dropna(),
    'Price'
)


Quantity 极端值检查
下界：-12.5
上界：23.5
极端值数量：116489
极端值占比：10.91%

Price 极端值检查
下界：-3.1000000000000005
上界：8.5
极端值数量：68105
极端值占比：6.38%


### 检查取消订单标记

In [15]:
# 转换为字符串，去除首尾空格
invoice = df['Invoice'].astype('string').str.strip()

# 判断是否以 C 开头
df['Is_Cancelled'] = invoice.str.upper().str.startswith(
    'C', na=False
)

# 统计取消订单记录
cancelled_rows = df[df['Is_Cancelled']]

print('========== 取消订单检查 ==========')

print('总记录数：', len(df))

print('C 开头的记录数：', len(cancelled_rows))

print('C 开头的唯一订单数：',
      cancelled_rows['Invoice'].nunique())

print('取消订单记录占比：',
      f'{len(cancelled_rows) / len(data) * 100:.2f}%')

display(cancelled_rows.head(10))


# 检查 C 开头记录中数量为负数的比例
print('C 开头且 Quantity 为负数：',
      (cancelled_rows['Quantity'] < 0).sum())

print('C 开头但 Quantity 非负数：',
      (cancelled_rows['Quantity'] >= 0).sum())

# 检查非 C 开头记录中数量为负数的情况
normal_rows = df[~df['Is_Cancelled']]

print('非 C 开头但 Quantity 为负数：',
      (normal_rows['Quantity'] < 0).sum())

invoice_check = df.groupby('Is_Cancelled').agg(
    记录数=('Invoice', 'size'),
    订单数=('Invoice', 'nunique'),
    负数量记录数=('Quantity', lambda x: (x < 0).sum()),
    零数量记录数=('Quantity', lambda x: (x == 0).sum())
)

display(invoice_check)


========== 取消订单检查 ==========
总记录数： 1067371
C 开头的记录数： 19494
C 开头的唯一订单数： 8292
取消订单记录占比： 974700.00%


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Is_Cancelled
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,True
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,True
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,True
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,True
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,True
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia,True
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia,True
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia,True
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,True
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom,True


C 开头且 Quantity 为负数： 19493
C 开头但 Quantity 非负数： 1
非 C 开头但 Quantity 为负数： 3457


,记录数,订单数,负数量记录数,零数量记录数
Is_Cancelled,,,,
False,1047877,45336,3457,0
True,19494,8292,19493,0


In [20]:
quality_records = [
    {
        '检查项目': '字段缺失',
        '检查方法': 'isnull()',
        '问题描述': '部分字段存在缺失值',
        '处理建议': '根据字段业务含义决定删除或填充'
    },
    {
        '检查项目': '重复记录',
        '检查方法': 'duplicated()',
        '问题描述': f'发现 {duplicate_count} 条重复记录',
        '处理建议': '确认重复原因后再决定是否删除'
    },
    {
        '检查项目': '负数数量',
        '检查方法': 'Quantity < 0',
        '问题描述': f'发现 {(df["Quantity"] < 0).sum()} 条负数量记录',
        '处理建议': '结合取消订单业务规则判断'
    },
    {
        '检查项目': '零数量',
        '检查方法': 'Quantity == 0',
        '问题描述': f'发现 {(df["Quantity"] == 0).sum()} 条零数量记录',
        '处理建议': '检查是否属于无效交易'
    },
    {
        '检查项目': '异常单价',
        '检查方法': 'Price < 0 或 Price == 0',
        '问题描述': f'发现 {(df["Price"] <= 0).sum()} 条非正单价记录',
        '处理建议': '检查赠品、退款或异常交易'
    },
    {
        '检查项目': '取消订单',
        '检查方法': 'Invoice 以 C 开头',
        '问题描述': f'发现 {len(cancelled_rows)} 条 C 开头记录',
        '处理建议': '验证本地数据是否适用该识别规则'
    }
]

quality_df = pd.DataFrame(quality_records)

display(quality_df)

,检查项目,检查方法,问题描述,处理建议
0,字段缺失,isnull(),部分字段存在缺失值,根据字段业务含义决定删除或填充
1,重复记录,duplicated(),发现 34335 条重复记录,确认重复原因后再决定是否删除
2,负数数量,Quantity < 0,发现 22950 条负数量记录,结合取消订单业务规则判断
3,零数量,Quantity == 0,发现 0 条零数量记录,检查是否属于无效交易
4,异常单价,Price < 0 或 Price == 0,发现 6207 条非正单价记录,检查赠品、退款或异常交易
5,取消订单,Invoice 以 C 开头,发现 19494 条 C 开头记录,验证本地数据是否适用该识别规则


In [24]:
from config.config import REPORT_DIR
# 保存数据质量报告
quality_df.to_csv(REPORT_DIR/'data_quality_report.csv', index=False)
print('数据质量报告保存成功！')

数据质量报告保存成功！
